# Embalses

### API SIMEM embalses ingestion y guardar json en raw files landing

In [0]:
!pip install pydataxm

In [0]:
df_embalses = ReadSIMEM('A0CF2A',"2026-06-01","2026-06-29").main(filter=False)

In [0]:
print(df_embalses.columns)

In [0]:
df_embalses_unicos = df_embalses.drop_duplicates(['CodigoEmbalse', 'NombreEmbalse'])
display(df_embalses_unicos)

## Obtener latitud y longitud de cada uno de los embalses

In [0]:
%pip install geopy

In [0]:
import pandas as pd
from geopy.geocoders import Nominatim
from time import sleep

geolocator = Nominatim(user_agent="simem_reservoir_geocoder")

def geocode_reservoir(row):
    codigo = row["CodigoEmbalse"]
    nombre = row["NombreEmbalse"]

    queries = [
        f"Embalse {nombre}, Colombia",
        f"Represa {nombre}, Colombia",
        f"Central Hidroeléctrica {nombre}, Colombia",
        f"{nombre}, Colombia",
        f"{codigo}, Colombia"
    ]

    for query in queries:
        try:
            location = geolocator.geocode(query, timeout=10)
            sleep(0.5)  # Reduce sleep to 0.5s to improve speed, but avoid rate limit issues

            if location:
                return pd.Series({
                    "LatitudGeopy": location.latitude,
                    "LongitudGeopy": location.longitude,
                    "GeocodingQuery": query,
                    "GeocodingStatus": "found"
                })
        except Exception as e:
            return pd.Series({
                "LatitudGeopy": None,
                "LongitudGeopy": None,
                "GeocodingQuery": query,
                "GeocodingStatus": f"error: {e}"
            })

    return pd.Series({
        "LatitudGeopy": None,
        "LongitudGeopy": None,
        "GeocodingQuery": " | ".join(queries),
        "GeocodingStatus": "not_found"
    })

# Use a cache to avoid repeated geocoding for the same embalse
geocode_cache = {}

def geocode_reservoir_with_cache(row):
    key = (row["CodigoEmbalse"], row["NombreEmbalse"])
    if key in geocode_cache:
        return geocode_cache[key]
    result = geocode_reservoir(row)
    geocode_cache[key] = result
    return result

df_coords_geopy = df_embalses_unicos.apply(geocode_reservoir_with_cache, axis=1)

df_embalses_geo = pd.concat(
    [df_embalses_unicos.reset_index(drop=True), df_coords_geopy],
    axis=1
)

fallback_coords = {
    "AGREGADO":  {"lat": 4.70, "lon": -73.90, "tipo": "representative"},
    "MIRATRON":  {"lat": 6.76, "lon": -75.28, "tipo": "representative"},
    "ALTOANCH":  {"lat": 3.55, "lon": -76.90, "tipo": "approximate"},
    "LAFE":      {"lat": 6.13, "lon": -75.50, "tipo": "approximate"},
    "EMBABOGO":  {"lat": 4.85, "lon": -73.88, "tipo": "representative"},
    "ESMERALD":  {"lat": 4.90, "lon": -73.35, "tipo": "approximate"},
    "ITUANGO":   {"lat": 7.13, "lon": -75.69, "tipo": "approximate"},
    "PORCE2":    {"lat": 6.79, "lon": -75.07, "tipo": "approximate"},
    "PORCE3":    {"lat": 6.85, "lon": -75.03, "tipo": "approximate"},
    "SANLOREN":  {"lat": 6.25, "lon": -74.88, "tipo": "approximate"},
    "SOGAMOSO":  {"lat": 6.83, "lon": -73.36, "tipo": "approximate"}
}

def complete_coordinates(row):
    codigo = row["CodigoEmbalse"]

    if pd.notna(row["LatitudGeopy"]) and pd.notna(row["LongitudGeopy"]):
        return pd.Series({
            "Latitud": row["LatitudGeopy"],
            "Longitud": row["LongitudGeopy"],
            "TipoCoordenada": "geopy",
            "CoordinateSource": "Nominatim"
        })

    fallback = fallback_coords.get(codigo)

    if fallback:
        return pd.Series({
            "Latitud": fallback["lat"],
            "Longitud": fallback["lon"],
            "TipoCoordenada": fallback["tipo"],
            "CoordinateSource": "fallback_manual"
        })

    return pd.Series({
        "Latitud": None,
        "Longitud": None,
        "TipoCoordenada": "missing",
        "CoordinateSource": "missing"
    })

df_completed_coords = df_embalses_geo.apply(complete_coordinates, axis=1)

df_dim_embalses = pd.concat(
    [df_embalses_geo, df_completed_coords],
    axis=1
)

df_dim_embalses = df_dim_embalses[
    [
        "CodigoEmbalse",
        "NombreEmbalse",
        "Latitud",
        "Longitud",
        "TipoCoordenada",
        "CoordinateSource",
        "GeocodingStatus",
        "GeocodingQuery"
    ]
]

display(df_dim_embalses)

display(
    df_dim_embalses[
        df_dim_embalses["Latitud"].isna() |
        df_dim_embalses["Longitud"].isna()
    ]
)

In [0]:
# Completar Riogrande II, que quedó sin coordenadas
df_dim_embalses.loc[
    df_dim_embalses["CodigoEmbalse"] == "RIOGRAN2",
    ["Latitud", "Longitud"]
] = [6.516009, -75.456121]

# Dejar solo columnas útiles para la dimensión
df_dim_embalses_final = df_dim_embalses[
    [
        "CodigoEmbalse",
        "NombreEmbalse",
        "Latitud",
        "Longitud"
    ]
].copy()

display(df_dim_embalses_final)

In [0]:
# Guardar como un solo archivo JSON usando pandas, sobrescribiendo si ya existe
df_dim_embalses_final.to_json("/Volumes/observatorio_dev/landing/raw_files/embalses_unicos.json", orient='records', lines=True, mode='w')

In [0]:
%sql
SELECT
    COUNT(*) AS registros,
    COUNT(DISTINCT codigo_embalse) AS embalses_distintos,
    SUM(
        CASE
            WHEN latitud IS NULL OR longitud IS NULL
              OR TRIM(latitud) = '' OR TRIM(longitud) = ''
            THEN 1 ELSE 0
        END
    ) AS registros_sin_coordenadas,
    MIN(load_date) AS primera_carga,
    MAX(load_date) AS ultima_carga
FROM observatorio_dev.bronze.embalses;

In [0]:
%sql
SELECT
    COUNT(*) AS grupos_duplicados,
    COALESCE(SUM(repeticiones), 0) AS filas_en_grupos,
    COALESCE(SUM(repeticiones - 1), 0) AS filas_excedentes,
    COALESCE(MAX(repeticiones), 0) AS maximo_repeticiones
FROM (
    SELECT
        codigo_embalse,
        nombre_embalse,
        latitud,
        longitud,
        COUNT(*) AS repeticiones
    FROM observatorio_dev.bronze.embalses
    GROUP BY
        codigo_embalse,
        nombre_embalse,
        latitud,
        longitud
    HAVING COUNT(*) > 1
);

In [0]:
%sql
SELECT
    codigo_embalse,
    COUNT(*) AS registros,
    COUNT(DISTINCT nombre_embalse) AS nombres_distintos,
    COUNT(DISTINCT CONCAT_WS('||', latitud, longitud))
        AS coordenadas_distintas
FROM observatorio_dev.bronze.embalses
GROUP BY codigo_embalse
HAVING COUNT(*) > 1
ORDER BY registros DESC;